In [1]:
import pandas as pd
import re
import string
import numpy as np
import os
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.callbacks import CallbackAny2Vec
from gensim.models.doc2vec import TaggedDocument

# Load poems

In [2]:
data = pd.read_csv('poem_data_with_birth.csv', engine='python')
data

,Title,Poet,Poem,Tags,Emotion,PoetBirth,Period
0,The New Church,Lucia Cherciu,"The old cupola glinted above the clouds, shone...",NaN,Fear,NaN,NaN
1,Look for Me,Ted Kooser,Look for me under the hood\r\r\nof that old Ch...,NaN,Sadness,1939.0,1941-1970
2,Wild Life,Grace Cavalieri,"Behind the silo, the Mother Rabbit\r\r\nhunche...",NaN,Surprise,1932.0,1941-1970
3,Umbrella,Connie Wanek,When I push your button\r\r\nyou fly off the h...,NaN,Fear,1952.0,1971-Present
4,Sunday,January Gill O'Neil,You are the start of the week\r\r\nor the end ...,NaN,Anger,NaN,NaN
...,...,...,...,...,...,...,...
13740,Heat Wave,Samuel Menashe,Sheets entangle him Naked on his bed Like ...,"The Body,Nature",NaN,1925.0,1941-1970
13741,Humidifier,Louise Glück,—After Robert Pinsky\r\r\n\r\r\n\r\r\n\r\r\nDe...,NaN,NaN,1943.0,1941-1970
13742,The Modern Pastoral Elegy,Conor O'Callaghan,A Tick-Where-Appropriate Template\r\r\n\r\r\n\...,"Living,Arts & Sciences,Humor & Satire,Poetry &...",NaN,NaN,NaN
13743,On the Metro,C. K. Williams,"On the metro, I have to ask a young woman to m...","Living,Growing Old,Love,Desire,Realistic & Com...",NaN,1936.0,1941-1970


In [3]:
poem_idx = 0
poem_example = data.iloc[poem_idx,2]
poem_example

'The old cupola glinted above the clouds, shone\r\r\namong fir trees, but it took him an hour\r\r\nfor the half mile all the way up the hill. As he trailed,\r\r\nthe village passed him by, greeted him,\r\r\nasked about his health, but everybody hurried\r\r\nto catch the mass, left him leaning against fences,\r\r\nmeasuring the road with the walking stick he sculpted.\r\r\nHe yearned for the day when the new church\r\r\nwould be built—right across the road. Now\r\r\nit rises above the moon: saints in frescoes\r\r\nmeet the eye, and only the rain has started to cut\r\r\nthrough the shingles on the roof of his empty\r\r\nhouse. The apple trees have taken over the sky,\r\r\nsequestered the gate, sidled over the porch.'

New lines notation is *\r\r\n*

# Train word2vec models on poem words

**Preprocessing:** 
- split to words
- don't keep punctuation
- keep line endings as special token *\<linesep\>* to map to the embedding space for structure representation
    - later give a fix weight to it in averaging?

In [4]:
LINESEP_RE = re.compile(r'\r+\n')
WHITESPACE_RE = re.compile(r'\s+')
TOKEN_RE = re.compile(r'<linesep>|\b\w+\b')

def preprocess(text):
    text = text.lower()
    text = LINESEP_RE.sub(' <linesep> ', text)
    text = WHITESPACE_RE.sub(' ', text)
    return TOKEN_RE.findall(text)

In [5]:
poems = data.iloc[:,2]

In [6]:
tokenized_poems = [preprocess(poem) for poem in poems]

In [7]:
poems[200], tokenized_poems[200]

("from Pia Arke's exhibition Arctic Hysteria at Greenland's \r\r\nNational Museum & Archives, Nuuk, 2010\r\r\n\r\r\n\r\r\n\r\r\ni.\r\r\r\nI am in my body. I am here, in front of you. I am the temperature in this room. I am undressed in my nudity; I am the light and shade you feel. I am more like other people than like you. I have before and after. I am my self, entirely and only. My outside and inside are continuous. I am muscle, organ, fluid, bone. I am monumental. You are the only one who sees me. ii. \r\r\r\nI am not naked as I am; I am naked as you see me. I am transparent, almost visible. I have a time and a place. I am tribal and exotic. I must always carry objects. You are heroic. I am a complete museum, the story of my own making. I am a mirror to you; you are reflected in the looking at me. At best, I mimic you. You write me. When you leave, I will no longer exist. iii. \r\r\r\nI am a single conscious point. I am indifferent. I am unself, like a photogram. I am prehistoric, be

## Number of word tokens in total

In [8]:
# Flatten the list of lists into a single list of tokens
all_tokens = [token for poem in tokenized_poems for token in poem]

# Count total tokens
total_tokens = len(all_tokens)
vocab_size = len(set(all_tokens))
total_tokens, vocab_size

(3905594, 105836)

100 dimensions for the embedding should be alright

## Train model

In [8]:
EMBEDDING_DIM = 100 # size of embedding vectors
WINDOW = 5 # context window size
SKIPGRAM = 0 # 1 = skip-gram, 0 = CBOW

In [9]:
class EpochLogger(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
    def on_epoch_end(self, model):
        self.epoch += 1
        print(f"Epoch {self.epoch} end")

In [16]:
epoch_logger = EpochLogger()
model_w2v = Word2Vec(
    sentences=tokenized_poems,
    vector_size=EMBEDDING_DIM,
    window=WINDOW,
    min_count=3, # ignore words appearing less than this, necessary because of messed up words in the dataset
    workers=4,
    sg=SKIPGRAM,
    epochs=20,
    callbacks = [epoch_logger]
)
model_w2v.save(f"models/w2v_d{EMBEDDING_DIM}_w{WINDOW}_{'sg' if SKIPGRAM else 'cbow'}.model")

Epoch 1 end
Epoch 2 end
Epoch 3 end
Epoch 4 end
Epoch 5 end
Epoch 6 end
Epoch 7 end
Epoch 8 end
Epoch 9 end
Epoch 10 end
Epoch 11 end
Epoch 12 end
Epoch 13 end
Epoch 14 end
Epoch 15 end
Epoch 16 end
Epoch 17 end
Epoch 18 end
Epoch 19 end
Epoch 20 end


In [18]:
model_w2v.wv.most_similar("<linesep>")

[('just', 0.4671189785003662),
 ('happening', 0.46017420291900635),
 ('tired', 0.45365118980407715),
 ('talking', 0.44727668166160583),
 ('history', 0.44072699546813965),
 ('today', 0.42807912826538086),
 ('sleeping', 0.42674657702445984),
 ('thinking', 0.4223511517047882),
 ('imagine', 0.4201611280441284),
 ('drowning', 0.4144587814807892)]

#### Skip-gram:
- Learns embeddings for rare words better
- With a small corpus, may overfit on rare words, producing strange neighbors
- Low-frequency words dominate “most similar” because skip-gram tries to predict context for every word, including very rare ones.
#### CBOW:
- Predicts a word from its context
- More robust on small datasets, especially for common words
- Gives much more intuitive semantic neighbors

# Train doc2vec model on full poem data

In [32]:
# id is poem index
tagged_poems = [
    TaggedDocument(words=poem_tokens, tags=[i])
    for i, poem_tokens in enumerate(tokenized_poems)
]

In [33]:
DISTRIBUTED_MEMORY = 0 # 1 - DM, 0 - DBOW

In [34]:
epoch_logger = EpochLogger()
model_d2v = Doc2Vec(
    documents=tagged_poems,
    vector_size=EMBEDDING_DIM,
    window=WINDOW,
    min_count=3,
    workers=4,
    epochs=20,
    dm=DISTRIBUTED_MEMORY,
    callbacks=[epoch_logger]
)

model_d2v.save(f"models/d2v_poem_d{EMBEDDING_DIM}_w{WINDOW}_{'dm' if DISTRIBUTED_MEMORY else 'dbow'}.model")

Epoch 1 end
Epoch 2 end
Epoch 3 end
Epoch 4 end
Epoch 5 end
Epoch 6 end
Epoch 7 end
Epoch 8 end
Epoch 9 end
Epoch 10 end
Epoch 11 end
Epoch 12 end
Epoch 13 end
Epoch 14 end
Epoch 15 end
Epoch 16 end
Epoch 17 end
Epoch 18 end
Epoch 19 end
Epoch 20 end


In [35]:
model_d2v.dv.most_similar(0, topn=5)

[(2111, 0.6589506268501282),
 (11487, 0.6416528820991516),
 (9719, 0.6407117247581482),
 (428, 0.6342081427574158),
 (1753, 0.633522629737854)]

In [37]:
poems[0], poems[428]

('The old cupola glinted above the clouds, shone\r\r\namong fir trees, but it took him an hour\r\r\nfor the half mile all the way up the hill. As he trailed,\r\r\nthe village passed him by, greeted him,\r\r\nasked about his health, but everybody hurried\r\r\nto catch the mass, left him leaning against fences,\r\r\nmeasuring the road with the walking stick he sculpted.\r\r\nHe yearned for the day when the new church\r\r\nwould be built—right across the road. Now\r\r\nit rises above the moon: saints in frescoes\r\r\nmeet the eye, and only the rain has started to cut\r\r\nthrough the shingles on the roof of his empty\r\r\nhouse. The apple trees have taken over the sky,\r\r\nsequestered the gate, sidled over the porch.',
 'The farmer in deep thought\r\r\nis pacing through the rain\r\r\namong his blank fields, with\r\r\nhands in pockets,\r\r\nin his head\r\r\nthe harvest already planted.\r\r\nA cold wind ruffles the water\r\r\namong the browned weeds.\r\r\nOn all sides\r\r\nthe world rolls 

# Train doc2vec model on lines of poem data

Id is in format: *[poem_idx]_[line_idx]*

In [10]:
tagged_lines = []

for poem_idx, poem_tokens in enumerate(tokenized_poems):
    current_line = []
    line_idx = 0
    for token in poem_tokens:
        if token == "<linesep>":
            if current_line:  # skip empty lines
                tag = f"{poem_idx}_{line_idx}"
                tagged_lines.append(TaggedDocument(words=current_line, tags=[tag]))
                current_line = []
                line_idx += 1
        else:
            current_line.append(token)
    
    if current_line:
        tag = f"{poem_idx}_{line_idx}"
        tagged_lines.append(TaggedDocument(words=current_line, tags=[tag]))

In [11]:
print(tagged_lines[:5])

[TaggedDocument(words=['the', 'old', 'cupola', 'glinted', 'above', 'the', 'clouds', 'shone'], tags=['0_0']), TaggedDocument(words=['among', 'fir', 'trees', 'but', 'it', 'took', 'him', 'an', 'hour'], tags=['0_1']), TaggedDocument(words=['for', 'the', 'half', 'mile', 'all', 'the', 'way', 'up', 'the', 'hill', 'as', 'he', 'trailed'], tags=['0_2']), TaggedDocument(words=['the', 'village', 'passed', 'him', 'by', 'greeted', 'him'], tags=['0_3']), TaggedDocument(words=['asked', 'about', 'his', 'health', 'but', 'everybody', 'hurried'], tags=['0_4'])]


In [15]:
DISTRIBUTED_MEMORY = 0 # 1 - DM, 0 - DBOW

epoch_logger = EpochLogger()
model_d2v_lines = Doc2Vec(
    documents=tagged_lines,
    vector_size=EMBEDDING_DIM,
    window=WINDOW,
    min_count=3,
    workers=4,
    epochs=20,
    dm=DISTRIBUTED_MEMORY,
    callbacks=[epoch_logger]
)

model_d2v_lines.save(f"models/d2v_line_d{EMBEDDING_DIM}_w{WINDOW}_{'dm' if DISTRIBUTED_MEMORY else 'dbow'}.model")

Epoch 1 end
Epoch 2 end
Epoch 3 end
Epoch 4 end
Epoch 5 end
Epoch 6 end
Epoch 7 end
Epoch 8 end
Epoch 9 end
Epoch 10 end
Epoch 11 end
Epoch 12 end
Epoch 13 end
Epoch 14 end
Epoch 15 end
Epoch 16 end
Epoch 17 end
Epoch 18 end
Epoch 19 end
Epoch 20 end


In [16]:
model_d2v_lines.dv.most_similar("0_0", topn=5)

[('11846_8', 0.8928831219673157),
 ('3968_55', 0.8791084885597229),
 ('2010_33', 0.8759530186653137),
 ('244_72', 0.8669785261154175),
 ('11604_2', 0.8631607294082642)]

In [17]:
poems[0], poems[11604]

('The old cupola glinted above the clouds, shone\r\r\namong fir trees, but it took him an hour\r\r\nfor the half mile all the way up the hill. As he trailed,\r\r\nthe village passed him by, greeted him,\r\r\nasked about his health, but everybody hurried\r\r\nto catch the mass, left him leaning against fences,\r\r\nmeasuring the road with the walking stick he sculpted.\r\r\nHe yearned for the day when the new church\r\r\nwould be built—right across the road. Now\r\r\nit rises above the moon: saints in frescoes\r\r\nmeet the eye, and only the rain has started to cut\r\r\nthrough the shingles on the roof of his empty\r\r\nhouse. The apple trees have taken over the sky,\r\r\nsequestered the gate, sidled over the porch.',
 'I never knew the earth had so much gold—\r\r\nThe fields run over with it, and this hill\r\r\nHoary and old,\r\r\nIs young with buoyant blooms that flame and thrill.\r\r\nSuch golden fires, such yellow—lo, how good\r\r\nThis spendthrift world, and what a lavish God!\r\r\